---
title: "2. Struktury danych, operatory i funkcje w R"
subtitle: Od pojedynczej wartości do samodzielnej analizy
---

# 2. Struktury danych, operatory i funkcje w R

W tym rozdziale nauczysz się wybierać strukturę danych, odczytywać i przekształcać jej elementy oraz zapisywać powtarzalne obliczenia jako funkcje. Punktem wyjścia są raporty *Podstawowe operacje w R* i *Podstawowe operacje w R — część 2*. Przykłady rozwijają ich tematykę, a zadania pozwalają samodzielnie sprawdzić zrozumienie.

Pracuj kolejno: przeczytaj objaśnienie, przewidź wynik, uruchom kod i wykonaj małe zadanie. Komórki korzystają z obiektów utworzonych wcześniej. Podstawowa część wymaga R co najmniej 4.1 (ze względu na `|>`); przykłady z pakietami `tibble` i `magrittr` są opcjonalne. Dane tworzymy w notebooku, więc nie trzeba pobierać pliku ani używać `attach()`.

**Na start — zadanie 2.1.** Opisz trzy cechy mieszkania, które można zebrać w badaniu. Które są liczbami, które kategoriami, a które tekstem? Zapisz jednostki cech liczbowych.

## 2.1. Obiekty, typy i brakujące wartości

Nazwa wskazuje obiekt przechowujący dane. Operator `<-` przypisuje wartość do nazwy. R rozróżnia wielkie i małe litery: `wiek` i `Wiek` to różne nazwy. Wybieraj nazwy opisowe, np. `cena_m2`, i unikaj nazw funkcji takich jak `c`, `mean` czy `t`.

**Typ** mówi, jak przechowywane są elementy; **klasa** opisuje, jak R ma traktować obiekt. `typeof()` sprawdza typ, `class()` klasę, a `str()` daje krótki opis struktury. Najczęściej użyjemy wartości logicznych (`logical`), całkowitych (`integer`), zmiennoprzecinkowych (`double`) i tekstowych (`character`). Istnieją też typy `complex` i `raw`. Zwykły zapis `25` tworzy liczbę typu double, a `25L` — integer.

In [ ]:
wiek_osoby <- 25L
cena <- 420000.50
miasto <- "Gdańsk"
czy_balkon <- TRUE
str(list(wiek_osoby, cena, miasto, czy_balkon))
typeof(wiek_osoby)
class(cena)

`NA` oznacza brak wartości, `NaN` nieokreślony wynik obliczenia, a `Inf` nieskończoność. `NULL` oznacza brak obiektu/składnika; nie jest pojedynczą brakującą obserwacją. Braki sprawdzamy przez `is.na()`, nie przez `== NA`. Usunięcie braków z obliczeń wymaga świadomej decyzji: wynik opisuje wtedy tylko dostępne obserwacje.

In [ ]:
pomiary <- c(12, NA, 18)
is.na(pomiary)
mean(pomiary)
mean(pomiary, na.rm = TRUE)  # 15, na podstawie dwóch obserwacji
length(NULL)               # 0

**Zadanie 2.2.** Utwórz `punkty <- c(10, 15, NA, 20)`. Sprawdź typ, długość i liczbę braków (`sum(is.na(punkty))`). Oblicz średnią dostępnych wyników i napisz, z ilu obserwacji korzysta.

## 2.2. Wektory: jedna cecha, wiele obserwacji

Wektor atomowy zawiera elementy jednego typu. `c()` łączy wartości, a `length()` podaje ich liczbę. R wykonuje wiele operacji element po elemencie, czyli **wektorowo**. Dodanie liczby do wektora zmienia każdy jego element.

In [ ]:
wiek <- c(15, 25, 40, 50, 60, 23, 33, 36, 19, 24)
length(wiek)
wiek + 1
seq(from = 10, to = 30, by = 5)
rep(c("A", "B"), each = 3)

Łączenie różnych typów może wymusić konwersję. W `c(20, "brak")` liczba staje się tekstem. `as.numeric()` może zamienić tekst liczbowy na liczbę, ale napis nieliczbowy spowoduje ostrzeżenie i `NA`. Dlatego braki liczb zapisujemy jako `NA`, a nie jako napis `"brak"`.

In [ ]:
mieszany <- c(20, "brak")
str(mieszany)
as.numeric(c("20", "30"))

**Zadanie 2.3.** Utwórz wektor pięciu cen w złotych. Przelicz je na tysiące złotych i oblicz ceny po obniżce o 10%. Wyjaśnij, dlaczego dopisanie `"PLN"` przez `c()` utrudni obliczenia.

### Wybieranie elementów

Indeksy w R zaczynają się od **1**. Nawiasy `[]` wybierają elementy według pozycji, nazw lub warunku. Indeks ujemny pomija wskazane pozycje; nie łącz indeksów dodatnich i ujemnych w jednym wyborze.

In [ ]:
wiek[c(1, 3)]
wiek[-1]
wiek[wiek > 30]
nazwane <- c(Ania = 20, Bartek = 22, Celina = 21)
nazwane["Bartek"]

**Zadanie 2.4.** Wybierz z `wiek` pierwsze trzy wartości, osoby w wieku co najmniej 30 lat oraz wszystkie wartości poza drugą. Ile osób spełnia warunek wieku?

## 2.3. Czynniki: dane kategoryczne

Czynnik (`factor`) opisuje przynależność do kategorii. Przechowuje kody całkowite i zestaw poziomów (`levels`). Kategorie mogą być nieuporządkowane (np. miasto) lub uporządkowane (np. niska, średnia, wysoka ocena). Same numery kodów nie są pomiarami ilościowymi.

Nawiązując do raportu, utworzymy sztuczną kategorię `ryzyko`: wiek powyżej 30 lat oznacza etykietę „wysokie”. **To wyłącznie reguła do nauki kodu, a nie model oceny ryzyka kredytowego.** `ifelse()` wybiera wynik osobno dla każdego elementu warunku.

In [ ]:
ryzyko <- factor(
  ifelse(wiek > 30, "wysokie", "niskie"),
  levels = c("niskie", "wysokie")
)
levels(ryzyko)
table(ryzyko)
as.integer(ryzyko)  # wewnętrzne kody poziomów, nie wiek

Pułapka: `as.numeric(factor(c("10", "20")))` zwraca kody poziomów, nie liczby 10 i 20. Dla tekstowych etykiet liczbowych konwersję wykonaj przez `as.numeric(as.character(obiekt))`.

**Zadanie 2.5.** Utwórz czynnik uporządkowany `ocena` z wartościami `"dobra", "słaba", "bardzo dobra", "dobra"`. Ustal kolejność od słabej do bardzo dobrej przez `levels` i `ordered = TRUE`. Wyświetl liczebności kategorii.

## 2.4. Ramka danych i tibble

Ramka danych (`data.frame`) opisuje obserwacje w wierszach i zmienne w kolumnach. Kolumny mogą mieć różne typy, ale tę samą liczbę obserwacji. Dla danych o osobach wiek jest liczbowy, a kategoria ryzyka jest czynnikiem.

In [ ]:
osoby <- data.frame(id = seq_along(wiek), wiek = wiek, ryzyko = ryzyko)
str(osoby)
head(osoby)
dim(osoby)
osoby$wiek
osoby[osoby$wiek > 30, c("id", "wiek")]
osoby[order(osoby$wiek, decreasing = TRUE), ]

W `osoby[wiersze, kolumny]` przecinek rozdziela dwa wymiary. Puste miejsce oznacza wybór wszystkich wierszy lub kolumn. `sort()` porządkuje wartości wektora, a `order()` zwraca kolejność pozycji — pozwala więc uporządkować całe wiersze bez rozdzielania danych tej samej osoby. `osoby["wiek"]` zachowuje ramkę, `osoby[["wiek"]]` zwraca kolumnę, a `osoby[, "wiek", drop = FALSE]` wymusza zachowanie dwóch wymiarów.

**Zadanie 2.6.** Dodaj kolumnę `pelnoletnia` z warunkiem `wiek >= 18`. Wybierz pełnoletnie osoby i posortuj je malejąco według wieku. Nie zmieniaj kolejności samej kolumny wieku.

**Tibble** jest odmianą ramki danych z pakietu `tibble`. Zwięźle drukuje dane wraz z typami kolumn. Wybór przez `[` zachowuje tibble, a `[[` pobiera pojedynczą kolumnę. Szczegóły tworzenia opisuje [dokumentacja tibble](https://tibble.tidyverse.org/reference/tibble.html). Współczesny `data.frame()` również domyślnie nie zamienia tekstów na czynniki.

In [ ]:
if (requireNamespace("tibble", quietly = TRUE)) {
  osoby_tbl <- tibble::as_tibble(osoby)
  print(osoby_tbl)
  print(osoby_tbl[, "wiek"])
  print(osoby_tbl[["wiek"]])
} else {
  message("Opcjonalnie zainstaluj pakiet tibble w konsoli: install.packages('tibble')")
}

**Zadanie 2.7.** Jeśli masz pakiet `tibble`, porównaj `class(osoby[, "wiek"])` i `class(osoby_tbl[, "wiek"])`. W obu strukturach pobierz wiek przez `[[`. Czy średnia jest taka sama?

## 2.5. Macierze i tablice wielowymiarowe

Macierz ma dwa wymiary. Typowa macierz liczbowa przechowuje elementy jednego typu, dlatego nadaje się do algebry liniowej. `matrix()` domyślnie wypełnia ją kolumnami; `byrow = TRUE` zmienia kolejność na wiersze. Macierz kwadratowa ma tyle samo wierszy i kolumn: wymiar 2 × 3 oznacza macierz prostokątną.

In [ ]:
B <- matrix(1:9, nrow = 3)
A <- matrix(1:6, nrow = 2, byrow = TRUE)
C <- matrix(1:6, nrow = 3, byrow = TRUE)
D <- rbind(A, c(1, 1, 1))
E <- cbind(C, c(1, 1, 1))
B
E[3, ]
D[, 3]
t(A)  # transpozycja: zamiana wierszy i kolumn

Tablica (`array`) może mieć więcej niż dwa wymiary. Poniżej przechowuje dwa pomiary dla trzech osób w dwóch grupach. Trzeci indeks wskazuje grupę. Wymiary 3 × 2 × 2 wymagają 12 wartości.

In [ ]:
tablica <- array(
  1:12, dim = c(3, 2, 2),
  dimnames = list(
    osoba = c("O1", "O2", "O3"),
    pomiar = c("przed", "po"),
    grupa = c("Grupa 1", "Grupa 2")
  )
)
tablica[, , "Grupa 1"]
tablica[, "po", "Grupa 2"]

Nie wkładaj wieku i tekstowych etykiet ryzyka do wspólnej macierzy liczbowej: wymuszona konwersja zmieni sposób przechowywania danych. Dla różnych typów zmiennych wybierz ramkę danych.

**Zadanie 2.8.** Utwórz macierz 2 × 3 z liczb 1–6 bez `byrow = TRUE` i porównaj ją z `A`. Z `tablica` wybierz pomiar „przed” dla wszystkich osób z obu grup. Jakie wymiary ma wynik?

## 2.6. Listy: różne obiekty w jednym miejscu

Lista może zawierać wektor, ramkę danych, tekst, a nawet kolejną listę. To dobry pojemnik na wyniki analizy. `[` zwraca podlistę, `[[` wydobywa jej pojedynczy składnik, a `$` wybiera składnik po nazwie.

In [ ]:
wyniki <- list(dane = osoby, srednia = mean(wiek), opis = "Próba ćwiczeniowa")
wyniki["srednia"]
wyniki[["srednia"]]
wyniki$dane$wiek

**Zadanie 2.9.** Utwórz listę z macierzą `A`, wektorem trzech ulubionych kolorów i pierwszymi trzema wiekami. Pobierz drugi kolor. Wskazówka: najpierw wybierz składnik listy, potem element wektora.

| Potrzeba | Struktura |
| :--- | :--- |
| Wartości jednej cechy tego samego typu | wektor atomowy |
| Kategorie i ich poziomy | czynnik |
| Prostokątna tabela obserwacji z różnymi typami kolumn | data.frame lub tibble |
| Prostokątny układ liczb do obliczeń macierzowych | macierz |
| Pomiary w kilku wymiarach | array |
| Zestaw różnych obiektów | lista |

## 2.7. Operatory arytmetyczne i przypisanie

Operator wykonuje działanie na swoich argumentach. `<-` zapisuje wynik; `==` porównuje wartości. W wywołaniu funkcji zapis `digits = 2` nadaje wartość argumentowi.

| Operator | Znaczenie | Przykład i wynik |
| :--- | :--- | :--- |
| `+`, `-`, `*`, `/` | podstawowe działania | `7 / 2` → 3,5 |
| `^` | potęgowanie | `2 ^ 3` → 8 |
| `%%` | reszta z dzielenia | `7 %% 2` → 1 |
| `%/%` | dzielenie całkowite | `7 %/% 2` → 3 |
| `:` | sekwencja z krokiem 1 lub −1 | `2:4` → 2, 3, 4 |
| `%*%` | iloczyn macierzowy | `A %*% C` → macierz 2 × 2 |

In [ ]:
c(7, 8, 9) %% 2
2 + 3 * 4
(2 + 3) * 4
A * A     # iloczyn odpowiadających sobie elementów
A %*% C   # liczba kolumn A musi równać się liczbie wierszy C

Nawiasy ułatwiają odczyt kolejności działań. Pisz `1:(n + 1)`, jeśli końcem sekwencji ma być `n + 1`.

**Recykling** polega na powtarzaniu krótszego wektora w obliczeniach. Dla liczby i wektora jest wygodny, ale dla dwóch wektorów różnej długości może ukryć pomyłkę. Gdy długość dłuższego nie jest wielokrotnością krótszego, R zwykle ostrzega. Brak ostrzeżenia nie gwarantuje sensowności obliczenia.

In [ ]:
c(10, 20, 30, 40) + c(1, 2)  # 11, 22, 31, 42

**Zadanie 2.10.** Oblicz reszty z dzielenia `wiek` przez 2 i wybierz parzyste wartości. Przewidź wynik `1:6 * c(1, 10)`, a potem sprawdź go w R.

## 2.8. Porównania i logika

Operatory `==`, `!=`, `<`, `>`, `<=`, `>=` zwracają wartości logiczne. `&` oznacza „i”, `|` oznacza „lub”, a `!` negację. `%in%` sprawdza przynależność każdego elementu do podanego zbioru.

In [ ]:
wiek >= 18 & wiek < 30
wiek[wiek >= 18 & wiek < 30]
c("Gdańsk", "Sopot", "Toruń") %in% c("Gdańsk", "Sopot", "Gdynia")
any(wiek < 18)   # czy choć jedna osoba spełnia warunek?
all(wiek >= 18)  # czy każda osoba spełnia warunek?

`&` i `|` działają elementowo na wektorach. `&&` i `||` służą do pojedynczych warunków i stosują krótkie spięcie: prawa strona jest oceniana tylko wtedy, gdy jest potrzebna do ustalenia wyniku. W aktualnym R podanie im wektora dłuższego niż jeden element jest błędem. Do sprowadzenia wielu warunków do jednego użyj `any()` lub `all()`. Zobacz [dokumentację operatorów logicznych](https://stat.ethz.ch/R-manual/R-devel/library/base/html/Logic.html).

In [ ]:
liczby <- numeric(0)
length(liczby) > 0 && mean(liczby) > 10  # średnia nie musi być obliczana
wiek_z_brakiem <- c(20, NA, 35)
wybor <- !is.na(wiek_z_brakiem) & wiek_z_brakiem >= 18
wiek_z_brakiem[wybor]

Porównanie z `NA` daje brak rozstrzygnięcia. `any()` i `all()` także mogą zwrócić `NA`; użycie `na.rm = TRUE` pomija braki. Dla liczb zmiennoprzecinkowych ścisłe `==` bywa mylące wskutek ograniczonej precyzji: podobieństwo wyników obliczeń można sprawdzić przez `isTRUE(all.equal(0.1 + 0.2, 0.3))`.

**Zadanie 2.11.** Wybierz osoby mające mniej niż 20 lub więcej niż 50 lat. Sprawdź, czy wszystkie wartości wieku są dodatnie. Wyjaśnij, czemu `wiek > 20 && wiek < 50` nie jest poprawnym filtrem wektora.

## 2.9. Funkcje wbudowane i argumenty

Funkcję wywołujemy przez nazwę i nawiasy, np. `mean(wiek)`. Argumenty określają dane i sposób działania. Nazwy argumentów zwiększają czytelność, szczególnie przy wartościach opcjonalnych. Pomoc otworzysz w konsoli przez `?mean`, a listę argumentów sprawdzisz przez `args(mean)`.

In [ ]:
sum(wiek)
mean(wiek)
median(wiek)
range(wiek)
sd(wiek)
quantile(wiek)
round(mean(wiek), digits = 2)
sqrt(c(4, 9, 16))
log(exp(1))

`sd()` oblicza odchylenie standardowe próby (z mianownikiem n − 1). `log()` domyślnie używa logarytmu naturalnego. `round(x, digits = 2)` zaokrągla do dwóch miejsc po przecinku; natomiast `options(digits = 3)` dotyczy zasad drukowania cyfr znaczących i nie oznacza trzech miejsc po przecinku.

Przydatne funkcje pomocnicze to `sort()`, `rev()`, `unique()`, `length()`, `names()`, `nrow()` i `ncol()`. Przed konwersją warto sprawdzić dane przez `is.numeric()` lub `is.character()`. Zapis `pakiet::funkcja()` wybiera eksportowaną funkcję z konkretnego pakietu, np. `stats::sd(wiek)`.

**Zadanie 2.12.** Oblicz średnią, medianę i odchylenie standardowe `wiek`. Zaokrąglij wyniki do jednego miejsca po przecinku i opisz jednostkę każdego wyniku. Wyświetl trzy największe wartości wieku.

## 2.10. Potoki: wynik jako wejście następnej funkcji

Potok, nazywany też „fajką”, ułatwia czytanie kolejnych przekształceń. W prostych wywołaniach `x |> f()` oznacza `f(x)`: lewa strona trafia do pierwszego argumentu funkcji. Bazowy `|>` jest dostępny od R 4.1 i nie wymaga pakietu.

In [ ]:
round(mean(wiek), digits = 1)
wiek |> mean() |> round(digits = 1)
osoby |> subset(wiek >= 18) |> head(n = 3)

Potok sam nie zapisuje zmian w danych. Użyj przypisania, gdy chcesz zachować wynik. W kodzie korzystającym z `magrittr` spotkasz `%>%`. W prostych przykładach pełni podobną rolę, ale bardziej zaawansowana składnia obu potoków różni się.

In [ ]:
if (requireNamespace("magrittr", quietly = TRUE)) {
  library(magrittr)
  print(wiek %>% mean() %>% round(digits = 1))
} else {
  message("Przykład opcjonalny wymaga pakietu magrittr.")
}

W raportach pojawiają się również operatory `magrittr`: `%T>%` wykonuje działanie po prawej, lecz przekazuje dalej obiekt z lewej (przydatne np. do podglądu), `%<>%` przypisuje przekształcony wynik z powrotem do obiektu, a `%$%` udostępnia nazwy jego składników w wyrażeniu. Zasady `%T>%` opisuje [dokumentacja magrittr](https://magrittr.tidyverse.org/reference/tee.html). Na początek wystarczy zwykły potok i jawne przypisanie.

**Zadanie 2.13.** Za pomocą `|>` posortuj `wiek` malejąco i wybierz pierwsze trzy wartości przez `head()`. Zapisz wynik jako `najstarsi`. Napisz równoważne wywołanie bez potoku.

## 2.11. Tekst i proste wyrażenia regularne

Tekst to dane, które często wymagają oczyszczenia. `paste()` łączy fragmenty z separatorem, `paste0()` bez separatora, `nchar()` liczy znaki, `trimws()` usuwa białe znaki z końców, a `tolower()` i `toupper()` zmieniają wielkość liter.

In [ ]:
nazwy <- c(" Gdansk ", "SOPOT", "Gdynia")
tolower(trimws(nazwy))
paste("Osoba", 1:3)
paste0("ID_", 1:3)
nchar("student")
substr("student", start = 1, stop = 3)

`grepl()` zwraca wektor logiczny dopasowań, `grep()` domyślnie ich pozycje, `sub()` zastępuje pierwsze dopasowanie w każdym napisie, a `gsub()` wszystkie. W wyrażeniu regularnym `^` oznacza początek, `$` koniec, `.` dowolny znak, a `*` powtórzenie poprzedniego elementu zero lub więcej razy. Dosłowną kropkę zapisujemy w napisie R jako `\\.`.

In [ ]:
emails <- c("anna@uczelnia.edu", "education@world.gov", "invalid.edu", "jan@college.edu")
wybrane <- grepl("@.+\\.edu$", emails)
emails[wybrane]
grep("@.+\\.edu$", emails)
sub("@.+$", "@pg.edu.pl", emails[wybrane])
sub("-", " ", "analiza-danych-R")
gsub("-", " ", "analiza-danych-R")

Ten wzorzec wybiera napisy z `@` i końcówką `.edu`; nie jest pełnym walidatorem adresów e-mail. Gdy szukasz dosłownego fragmentu, możesz zastosować `fixed = TRUE`, np. `grepl(".edu", emails, fixed = TRUE)`.

**Zadanie 2.14.** Utwórz cztery przykładowe adresy, w tym dwa kończące się `.pl`. Wybierz je przez `grepl()`, a następnie zamień ich domeny na `student.pg.edu.pl`. Porównaj wyniki `grep()` i `grepl()`.

## 2.12. Daty i czas

Napis `"2026-10-03"` nie jest jeszcze datą. Klasa `Date` służy do dat kalendarzowych, a `POSIXct` do dat z czasem. Jawny format wejścia zapobiega pomyleniu dnia z miesiącem; jawna strefa czasowa jest ważna przy momentach zdarzeń.

In [ ]:
daty <- as.Date(c("03.10.2026", "10.10.2026"), format = "%d.%m.%Y")
class(daty)
daty + 7
difftime(daty[2], daty[1], units = "days")
format(daty, "%Y/%m/%d")
moment <- as.POSIXct("2026-10-03 12:00:00", tz = "Europe/Warsaw")
format(moment, tz = "UTC", usetz = TRUE)
seq(daty[1], by = "2 weeks", length.out = 4)

`format()` zwraca tekst służący do prezentacji. Do obliczeń zachowuj obiekt daty/czasu. Dodanie 1 do `Date` dodaje dzień, a do `POSIXct` sekundę. Doba kalendarzowa w lokalnej strefie może mieć inną długość niż 24 godziny przy zmianie czasu. Pakiet `lubridate`, wykorzystany w raporcie, oferuje m.in. `ymd()`, `dmy()` i `floor_date()`; podstawy można przećwiczyć bez dodatkowych pakietów.

**Zadanie 2.15.** Zamień napisy `"01.10.2026"` i `"15.10.2026"` na daty, policz odstęp w dniach i utwórz cztery cotygodniowe terminy zajęć od pierwszej daty.

## 2.13. Własne funkcje

Funkcja nadaje nazwę obliczeniu, które ma działać dla różnych danych. Składa się z argumentów i ciała w `{}`. Wynikiem jest ostatnie obliczone wyrażenie lub wartość wskazana przez `return()`. Argumenty mogą mieć wartości domyślne.

In [ ]:
cena_po_rabacie <- function(cena, rabat = 0.10) {
  wynik <- cena * (1 - rabat)
  return(wynik)
}
cena_po_rabacie(c(100, 200))
cena_po_rabacie(200, rabat = 0.25)

`wynik` jest zmienną lokalną, utworzoną podczas wywołania. Funkcja może też odczytywać zmienne z otaczającego środowiska, ale przekazywanie potrzebnych danych przez argumenty ułatwia odtworzenie obliczeń. Zwracana wartość i tekst wydrukowany przez `print()` lub `cat()` to różne rzeczy — zwrócony wynik można wykorzystać dalej.

**Zadanie 2.16.** Napisz funkcję `cena_za_m2(cena, powierzchnia)`. Sprawdź ją dla mieszkania za 400000 zł o powierzchni 50 m² i dla dwóch wektorów tej samej długości. Co stanie się dla powierzchni równej zero?

### Warunki i sprawdzanie danych

`if` wymaga jednego warunku logicznego bez `NA`. `else` wybiera alternatywną ścieżkę. `ifelse()` wybiera wartości elementowo dla całego wektora. Nie są to zamienne konstrukcje.

In [ ]:
ocen_srednia <- function(x) {
  if (!is.numeric(x) || length(x) == 0 || anyNA(x)) {
    stop("Podaj niepusty wektor liczbowy bez braków.")
  }
  if (mean(x) >= 50) {
    "Średnia co najmniej 50"
  } else {
    "Średnia poniżej 50"
  }
}
ocen_srednia(c(40, 60))
ifelse(wiek >= 18, "pełnoletnia", "niepełnoletnia")

**Zadanie 2.17.** Rozszerz `cena_za_m2()` o sprawdzenie, czy wszystkie powierzchnie są dodatnie i czy nie ma braków. W razie niespełnienia warunku użyj `stop()` z czytelnym komunikatem. Sprawdź poprawne dane, zero i `NA` w osobnych wywołaniach w konsoli.

### Przykład: standaryzacja

Standaryzacja przekształca pomiar według wzoru **z = (x − średnia) / odchylenie standardowe**. Wynik informuje, o ile odchyleń standardowych obserwacja odbiega od średniej. Nie ma jednostki pierwotnej zmiennej. Dla co najmniej dwóch skończonych obserwacji i dodatniego odchylenia przekształcony wektor ma średnią bliską 0 oraz odchylenie standardowe bliskie 1. Nie oznacza to uzyskania rozkładu normalnego.

In [ ]:
standaryzuj <- function(x, na.rm = FALSE) {
  if (!is.numeric(x)) stop("x musi być wektorem liczbowym.")
  if (!is.logical(na.rm) || length(na.rm) != 1 || is.na(na.rm)) {
    stop("na.rm musi mieć wartość TRUE albo FALSE.")
  }
  if (anyNA(x) && !na.rm) stop("Dane zawierają braki; rozważ na.rm = TRUE.")
  dostepne <- x[!is.na(x)]
  if (length(dostepne) < 2 || any(!is.finite(dostepne))) {
    stop("Potrzeba co najmniej dwóch skończonych obserwacji.")
  }
  odchylenie <- sd(dostepne)
  if (odchylenie == 0) stop("Nie można standaryzować stałej zmiennej.")
  (x - mean(dostepne)) / odchylenie
}
wiek_z <- standaryzuj(wiek)
mean(wiek_z)
sd(wiek_z)
standaryzuj(c(10, NA, 30), na.rm = TRUE)  # brak pozostaje na swoim miejscu

**Zadanie 2.18.** Zestandaryzuj `c(100, 200, 300, 400)`. Wyjaśnij znak wyniku dla pierwszej obserwacji. Osobno sprawdź reakcję funkcji na `c(5, 5, 5)`, pojedynczą liczbę i same braki. Dlaczego te przypadki wymagają innego postępowania?

## 2.14. Powtarzanie obliczeń

Pętla `for` wykonuje kod dla kolejnych elementów. `while` powtarza go, dopóki warunek jest prawdziwy — trzeba zapewnić zmianę prowadzącą do zakończenia. Dla prostych działań na wektorach zwykle wystarcza zapis wektorowy. `lapply()` stosuje funkcję do składników i zwraca listę, a `vapply()` dodatkowo wymaga zadeklarowania kształtu wyniku.

In [ ]:
for (i in seq_along(wiek[1:3])) {
  print(paste("Osoba", i, "ma", wiek[i], "lat"))
}
licznik <- 1
while (licznik <= 3) {
  print(licznik)
  licznik <- licznik + 1
}
podsumowania <- lapply(list(wiek = wiek, wiek_z = wiek_z), summary)
podsumowania
vapply(list(wiek = wiek, wiek_z = wiek_z), mean, numeric(1))

`seq_along(x)` jest bezpiecznym sposobem tworzenia indeksów: dla pustego `x` daje pusty ciąg. `1:length(x)` dla pustego wektora daje natomiast `c(1, 0)`.

**Zadanie 2.19.** Użyj `lapply()` do obliczenia średniej wektorów `c(2, 4, 6)` i `c(10, 20, 30)` zapisanych w liście. Powtórz obliczenia przez `vapply()` i porównaj struktury wyników.

## 2.15. Mały projekt: dane o mieszkaniach

Połącz poznane narzędzia w krótkiej analizie. Poniższy zbiór jest **syntetyczny**, inspirowany tematyką transakcji nieruchomości z raportów; nie przedstawia danych rynkowych Sacramento ani rzeczywistych cen w Polsce.

In [ ]:
mieszkania <- data.frame(
  id = 1:8,
  typ = factor(c("mieszkanie", "dom", "mieszkanie", "mieszkanie",
                 "dom", "mieszkanie", "dom", "mieszkanie")),
  pokoje = c(2L, 4L, 2L, 3L, 5L, 1L, 4L, 2L),
  powierzchnia = c(45, 120, 50, 68, 150, 30, 110, 48),
  cena = c(450000, 900000, 520000, 680000, 1200000, 330000, 850000, NA)
)
str(mieszkania)

**Zadanie 2.20 — wykonaj etapami.**

1. Podaj liczbę obserwacji, typy kolumn i liczbę brakujących cen. Wyjaśnij, dlaczego `id` mimo liczbowego zapisu nie jest cechą do uśredniania.
2. Dodaj `cena_m2`, korzystając z własnej funkcji. Posortuj całe wiersze według tej wartości, a brak umieść na końcu (`na.last = TRUE`).
3. Wybierz mieszkania dwupokojowe o znanej cenie. Oblicz ich średnią cenę oraz średnią cenę za m². Podaj jednostki.
4. Dodaj `cena_z` przez `standaryzuj(mieszkania$cena, na.rm = TRUE)`. Sprawdź średnią i odchylenie dla dostępnych wyników.
5. Narysuj osobno wykresy pudełkowe `cena` i `cena_z`. Możesz ustawić `par(mfrow = c(1, 2))`, a po rysowaniu przywrócić `par(mfrow = c(1, 1))`. Opisz zmianę skali i zachowanie kolejności obserwacji.
6. Zapisz ramkę oraz obie średnie w nazwanej liście. Jeśli masz `tibble`, utwórz również wersję tabeli w tej klasie.

**Punkty kontrolne:** zbiór ma 8 wierszy i jedną brakującą cenę. Filtr z punktu 3 powinien wybrać identyfikatory 1 i 3. Średnia cena tych lokali to 485000 zł. Dla `cena_z` średnia powinna być bliska 0, a odchylenie bliskie 1; brak pozostaje brakiem. Sama cena zapisana jako integer może być poprawna dla pełnych złotych mieszczących się w zakresie tego typu; double pozwala reprezentować także części złotego, ale używa przybliżonej reprezentacji binarnej.

## 2.16. Sprawdź rozumienie

**Zadanie 2.21 — odpowiedz bez uruchamiania kodu.**

- Czym różni się lista od wektora atomowego?
- Dlaczego sortowanie jednej kolumny i wstawienie jej do niezmienionej ramki może uszkodzić dane?
- Co zwracają `wyniki["dane"]` oraz `wyniki[["dane"]]`?
- Kiedy użyjesz `&`, a kiedy `&&`?
- Dlaczego `NA == NA` nie służy do wykrywania braków?
- Dlaczego `A * A` i `A %*% C` opisują różne działania?
- Jakie dane uniemożliwiają standaryzację i dlaczego?

Wróć do przykładów przy pytaniach, na które nie umiesz jeszcze odpowiedzieć. Na koniec uruchom notebook od początku w czystej sesji R i sprawdź, czy własne rozwiązania nie zależą od przypadkowych obiektów z wcześniejszej pracy.

## Materiały źródłowe i dalsza lektura

- **raport1.Rmd**, *Podstawowe operacje w R* — inspiracja do ćwiczeń o wieku, czynnikach, ramkach, macierzach, tablicach i listach oraz wprowadzenia do potoków.
- **raport2.Rmd**, *Podstawowe operacje w R — część 2. Operatory i funkcje* — zakres operatorów, tekstu, dat, funkcji, sterowania wykonaniem, tibble i zadania ze standaryzacją. Przykłady zostały opracowane ponownie i uzupełnione o objaśnienia oraz kontrolę danych.
- [R Core Team, An Introduction to R](https://cran.r-project.org/doc/manuals/r-release/R-intro.html) — podręcznik do dalszej nauki, zwłaszcza rozdziały 2–6 oraz 9–10.
- [Operatory logiczne w R](https://stat.ethz.ch/R-manual/R-devel/library/base/html/Logic.html) — działanie operatorów wektorowych i krótkiego spięcia.
- [Dokumentacja tibble](https://tibble.tidyverse.org/reference/tibble.html) i [operatora tee w magrittr](https://magrittr.tidyverse.org/reference/tee.html) — materiały do części opcjonalnej.

[← Wróć do instalacji i konfiguracji R oraz RStudio](rozdzial-1.ipynb)

[Przejdź do osobnego podrozdziału 2.17: formatowanie wyników z kable →](rozdzial-2-kable.ipynb)